# Step 5D ? Session-aware class-weight calibration for v5

The session-aware v4 model improved the held-out recovery-session result, but failed a fresh live shadow traversal because it over-predicted turns during true-forward travel.

Live v4 shadow evidence:

- agreement: 84.65%
- forward recall: 83.90%
- left recall: 81.58%
- right recall: 100.00%
- forward -> left: 38 / 789 = 4.82%
- forward -> right: 89 / 789 = 11.28%
- total forward false-turn rate: 16.10%
- longest disagreement streak: 20 frames

This notebook changes one controlled variable only: class-weight strength.

The data package, session split, 50?50 RGB preprocessing, CNN architecture, action classes, random seed, optimizer, and early-stopping rule remain fixed.


## Selection rule fixed before training

Candidate class-weight alphas:

- 0.75 ? v4 strength
- 0.50
- 0.25
- 0.00 ? no class weighting

Candidate selection uses recovery_02 validation data only.

A candidate is eligible only if both left recall and right recall are at least 70%.

Among eligible candidates, choose deterministically by:

1. shortest longest consecutive forward false-turn streak;
2. lowest total forward false-turn rate;
3. highest macro-F1.

The recovery_04 reference set is evaluated only after one candidate has been selected. Because recovery_04 was already inspected for v4, it is a repeat reference rather than a pristine final estimate. A fresh live shadow run remains the decisive external gate.


In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intelligent_Robotics_Design_Project"
SESSION_DATASET_ZIP_NAME = "step05c_session_dataset_v1.zip"

OUTPUT_MODEL_NAME = "maze_navigation_model_v5.h5"
OUTPUT_METRICS_NAME = "step05d_v5_metrics.json"
OUTPUT_CANDIDATES_NAME = "step05d_v5_candidate_validation_metrics.csv"
OUTPUT_CONFUSION_NAME = "step05d_v5_reference_confusion.csv"
OUTPUT_CONFIG_NAME = "step05d_v5_selected_config.json"

RANDOM_SEED = 42
IMAGE_SIZE = (50, 50)
BATCH_SIZE = 32
EPOCHS = 30
CANDIDATE_WEIGHT_ALPHAS = [0.75, 0.50, 0.25, 0.00]
MIN_LEFT_RECALL = 0.70
MIN_RIGHT_RECALL = 0.70

CLASS_NAMES = ("forward", "left", "right")
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

EXPECTED_SPLIT_SESSIONS = {
    "train": {"original_01", "original_02", "original_03", "recovery_01", "recovery_03"},
    "validation": {"recovery_02"},
    "test": {"recovery_04"},
}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

tf.config.experimental.enable_op_determinism()

project_dir = Path(DRIVE_PROJECT_DIR)
zip_path = project_dir / SESSION_DATASET_ZIP_NAME
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the dataset ZIP here first: {zip_path}")

print(f"TensorFlow: {tf.__version__}")
print(f"Dataset ZIP: {zip_path}")


In [ ]:
extract_root = Path("/content/step05d_session_dataset")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

manifest = pd.read_csv(extract_root / "manifest.csv")
required_columns = {"session_id", "split", "label", "relative_path", "source_filename"}
if set(manifest.columns) != required_columns:
    raise ValueError(f"Unexpected manifest columns: {list(manifest.columns)}")

manifest["label_index"] = manifest["label"].map(CLASS_TO_INDEX).astype(np.int32)
manifest["image_path"] = manifest["relative_path"].map(lambda value: str(extract_root / value))

for split, expected_sessions in EXPECTED_SPLIT_SESSIONS.items():
    actual_sessions = set(manifest.loc[manifest["split"] == split, "session_id"])
    if actual_sessions != expected_sessions:
        raise ValueError(f"{split} session mismatch: expected {sorted(expected_sessions)}, got {sorted(actual_sessions)}")

for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    if not EXPECTED_SPLIT_SESSIONS[a].isdisjoint(EXPECTED_SPLIT_SESSIONS[b]):
        raise ValueError(f"Session leakage between {a} and {b}")

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

print("SESSION ISOLATION CHECK: PASS")
print(f"train={len(train_df)}, validation={len(val_df)}, reference_test={len(test_df)}")


In [ ]:
def load_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32) / 255.0

def load_example(path, label):
    return load_image(path), label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame["image_path"].to_numpy(), frame["label_index"].to_numpy(dtype=np.int32))
    )
    dataset = dataset.map(load_example, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    if training:
        dataset = dataset.shuffle(len(frame), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)


In [ ]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def class_weights_for_alpha(labels, alpha):
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    balanced = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(CLASS_NAMES)),
        y=labels,
    ).astype(np.float64)
    raw = np.power(balanced, alpha)
    mean_sample_weight = float(np.dot(counts, raw) / counts.sum())
    normalized = raw / mean_sample_weight
    return {i: float(normalized[i]) for i in range(len(CLASS_NAMES))}

def longest_forward_false_turn_streak(labels, predictions):
    best = 0
    current = 0
    forward = CLASS_TO_INDEX["forward"]
    for human, predicted in zip(labels, predictions):
        if human == forward and predicted != forward:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return int(best)

def evaluate_predictions(frame, probabilities):
    labels = frame["label_index"].to_numpy(dtype=np.int32)
    predictions = np.argmax(probabilities, axis=1)
    cm = confusion_matrix(labels, predictions, labels=np.arange(3))
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=np.arange(3), zero_division=0
    )
    forward = CLASS_TO_INDEX["forward"]
    left = CLASS_TO_INDEX["left"]
    right = CLASS_TO_INDEX["right"]
    false_left = int(cm[forward, left])
    false_right = int(cm[forward, right])
    forward_support = int(cm[forward].sum())
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(np.mean(f1)),
        "precision_forward": float(precision[forward]),
        "precision_left": float(precision[left]),
        "precision_right": float(precision[right]),
        "recall_forward": float(recall[forward]),
        "recall_left": float(recall[left]),
        "recall_right": float(recall[right]),
        "support_forward": int(support[forward]),
        "support_left": int(support[left]),
        "support_right": int(support[right]),
        "forward_to_left": false_left,
        "forward_to_right": false_right,
        "forward_false_turn_rate": float((false_left + false_right) / forward_support),
        "longest_forward_false_turn_streak": longest_forward_false_turn_streak(labels, predictions),
        "confusion_matrix": cm,
    }


In [ ]:
train_labels = train_df["label_index"].to_numpy(dtype=np.int32)
candidate_rows = []
candidate_model_paths = {}

for alpha in CANDIDATE_WEIGHT_ALPHAS:
    print()
    print("=" * 72)
    print(f"Training candidate alpha={alpha:.2f}")
    print("=" * 72)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_SEED)

    train_ds = make_dataset(train_df, training=True)
    model = build_model()
    weights = class_weights_for_alpha(train_labels, alpha)
    print("Class weights:", {CLASS_NAMES[i]: round(weights[i], 4) for i in range(3)})

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=weights,
        callbacks=[early_stopping],
        verbose=2,
    )

    val_probabilities = model.predict(val_ds, verbose=0)
    metrics = evaluate_predictions(val_df, val_probabilities)
    cm = metrics.pop("confusion_matrix")

    eligible = (
        metrics["recall_left"] >= MIN_LEFT_RECALL
        and metrics["recall_right"] >= MIN_RIGHT_RECALL
    )

    row = {
        "alpha": alpha,
        "epochs_completed": len(history.history["loss"]),
        "eligible": bool(eligible),
        **metrics,
    }
    candidate_rows.append(row)

    candidate_path = Path(f"/content/step05d_candidate_alpha_{alpha:.2f}.keras")
    model.save(candidate_path)
    candidate_model_paths[alpha] = candidate_path

    print(
        f"validation accuracy={metrics['accuracy']:.2%}, "
        f"macro-F1={metrics['macro_f1']:.4f}, "
        f"left recall={metrics['recall_left']:.2%}, "
        f"right recall={metrics['recall_right']:.2%}"
    )
    print(
        f"forward false-turn rate={metrics['forward_false_turn_rate']:.2%}, "
        f"longest forward false-turn streak={metrics['longest_forward_false_turn_streak']}"
    )
    display(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

candidate_df = pd.DataFrame(candidate_rows)
display(candidate_df)


In [ ]:
eligible_df = candidate_df[candidate_df["eligible"]].copy()
if eligible_df.empty:
    raise RuntimeError(
        "No candidate met the fixed left/right recall floor. Stop here; do not evaluate recovery_04."
    )

selected_row = eligible_df.sort_values(
    by=[
        "longest_forward_false_turn_streak",
        "forward_false_turn_rate",
        "macro_f1",
    ],
    ascending=[True, True, False],
).iloc[0]

SELECTED_ALPHA = float(selected_row["alpha"])
print("SELECTED ALPHA:", SELECTED_ALPHA)
print("Selection row:")
display(selected_row.to_frame().T)

selected_model = tf.keras.models.load_model(candidate_model_paths[SELECTED_ALPHA])


## Repeat reference evaluation

Only the selected candidate is evaluated below on recovery_04.

This session remains excluded from candidate selection. It has already been inspected during the v4 experiment, so this is a repeat reference check, not a claim of a pristine untouched final test.


In [ ]:
reference_probabilities = selected_model.predict(test_ds, verbose=0)
reference_metrics = evaluate_predictions(test_df, reference_probabilities)
reference_confusion = reference_metrics.pop("confusion_matrix")

print(f"REFERENCE accuracy: {reference_metrics['accuracy']:.2%}")
print(f"REFERENCE macro-F1: {reference_metrics['macro_f1']:.4f}")
print(
    "REFERENCE recall:",
    {
        "forward": f"{reference_metrics['recall_forward']:.2%}",
        "left": f"{reference_metrics['recall_left']:.2%}",
        "right": f"{reference_metrics['recall_right']:.2%}",
    },
)
print(
    f"REFERENCE forward false-turn rate: {reference_metrics['forward_false_turn_rate']:.2%}; "
    f"longest streak: {reference_metrics['longest_forward_false_turn_streak']}"
)
display(pd.DataFrame(reference_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))


In [ ]:
project_dir.mkdir(parents=True, exist_ok=True)

model_path = project_dir / OUTPUT_MODEL_NAME
metrics_path = project_dir / OUTPUT_METRICS_NAME
candidates_path = project_dir / OUTPUT_CANDIDATES_NAME
confusion_path = project_dir / OUTPUT_CONFUSION_NAME
config_path = project_dir / OUTPUT_CONFIG_NAME

selected_model.save(model_path)
candidate_df.to_csv(candidates_path, index=False)
pd.DataFrame(reference_confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(confusion_path)

selected_weights = class_weights_for_alpha(train_labels, SELECTED_ALPHA)
config_payload = {
    "selected_alpha": SELECTED_ALPHA,
    "selection_rule": [
        "left recall >= 0.70 and right recall >= 0.70",
        "minimize longest forward false-turn streak",
        "minimize forward false-turn rate",
        "maximize macro-F1",
    ],
    "class_weights": {CLASS_NAMES[i]: selected_weights[i] for i in range(3)},
    "selected_validation": selected_row.to_dict(),
}
config_path.write_text(json.dumps(config_payload, indent=2), encoding="utf-8")

metrics_payload = {
    "method": "session-aware class-weight calibration",
    "random_seed": RANDOM_SEED,
    "image_size": list(IMAGE_SIZE),
    "candidate_alphas": CANDIDATE_WEIGHT_ALPHAS,
    "selected_alpha": SELECTED_ALPHA,
    "minimum_turn_recall": {
        "left": MIN_LEFT_RECALL,
        "right": MIN_RIGHT_RECALL,
    },
    "split_sessions": {key: sorted(value) for key, value in EXPECTED_SPLIT_SESSIONS.items()},
    "selected_validation": selected_row.to_dict(),
    "reference_recovery_04": reference_metrics,
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

print(f"Saved v5 model: {model_path}")
print(f"Saved metrics: {metrics_path}")
print(f"Saved candidate table: {candidates_path}")
print(f"Saved reference confusion: {confusion_path}")
print(f"Saved selected config: {config_path}")


## Gate after v5

Do not start autonomous motor control from these offline results.

Next:

1. place maze_navigation_model_v5.h5 in the local models folder;
2. verify the model contract locally;
3. run a fresh human-controlled live shadow traversal;
4. compare forward false-turn rate and longest false-turn streak against v4;
5. only if the fresh live shadow result is materially improved should autonomous Step 6 be reconsidered.

No controller heuristic should be used to conceal a model-level false-turn defect.
